# 7.1 Kaggle and Competitions

K-fold cross-validation, ensemble stacking meta-learner, and leaderboard shake-up simulation.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path

Path('output').mkdir(exist_ok=True)
rng = np.random.default_rng(42)

def sigmoid(x): return 1/(1+np.exp(-np.clip(x,-30,30)))
def log_loss(y,p): p=np.clip(p,1e-7,1-1e-7); return -np.mean(y*np.log(p)+(1-y)*np.log(1-p))
def accuracy(y,p,t=0.5): return np.mean((p>=t)==y)

n, d = 500, 12
X = rng.standard_normal((n, d))
w_true = rng.standard_normal(d)*0.5
y = (sigmoid(X@w_true + rng.normal(0,.5,n))>0.5).astype(float)

# 5-fold CV
idx = rng.permutation(n); folds = np.array_split(idx, 5)
scores = []
for i, val_idx in enumerate(folds):
    tr_idx = np.concatenate([folds[j] for j in range(5) if j!=i])
    Xtr,Xv,ytr,yv = X[tr_idx],X[val_idx],y[tr_idx],y[val_idx]
    w = np.zeros(d)
    for _ in range(100): w -= 0.1*(Xtr.T@(sigmoid(Xtr@w)-ytr)/len(ytr))
    scores.append(accuracy(yv, sigmoid(Xv@w)))
print(f'5-Fold CV: {[f"{s:.3f}" for s in scores]}')
print(f'Mean ± Std: {np.mean(scores):.3f} ± {np.std(scores):.3f}')

In [ ]:
preds = [sigmoid(X@rng.standard_normal(d)*s) for s in [0.3, 0.5, 0.8]]
W = np.ones(3)/3; losses = []
for _ in range(100):
    p = sigmoid(np.array(preds).T@W)
    losses.append(log_loss(y,p))
    err = p-y; g = np.array([np.mean(err*q) for q in preds])
    W -= 0.1*g; W = np.clip(W,0,None); W /= W.sum()+1e-10
print(f'Ensemble weights: {W.round(3)}, log-loss: {losses[-1]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(range(1,6), scores, color='steelblue')
axes[0].axhline(np.mean(scores), color='red', linestyle='--', label=f'Mean={np.mean(scores):.3f}')
axes[0].set(xlabel='Fold', ylabel='Accuracy', title='5-Fold CV'); axes[0].legend(); axes[0].set_ylim(0,1)
axes[1].plot(losses, color='tomato', lw=2)
axes[1].set(xlabel='Step', ylabel='Log-Loss', title='Ensemble Stacking Loss'); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('output/kaggle_competition.png')
print('Saved kaggle_competition.png')